# Credit Cluster Scoring Tool

Use this notebook to score a private, simulated, or competitor company against the saved credit clustering model.

Expected workflow:

1. Put the saved artifact at `saved_models/credit_cluster_model_v1.joblib`.
2. Enter company financials manually, or load a CSV/Excel file.
3. Run the scoring cells.
4. Review cluster assignment, near-default affinity, feature coverage, and warning flags.

Important: the output is **not a formal credit rating** and not a calibrated probability of default. It is a public-company financial profile match based on distance to the learned KMeans credit clusters.

In [8]:
import sys
import sklearn
import joblib
import numpy
import scipy

print("Python:", sys.executable)
print("sklearn:", sklearn.__version__, sklearn.__file__)
print("joblib:", joblib.__version__, joblib.__file__)
print("numpy:", numpy.__version__, numpy.__file__)
print("scipy:", scipy.__version__, scipy.__file__)

Python: C:\Users\kamen.dimitrov\AppData\Local\miniconda3\python.exe
sklearn: 1.9.0 C:\Users\kamen.dimitrov\AppData\Roaming\Python\Python313\site-packages\sklearn\__init__.py
joblib: 1.5.3 C:\Users\kamen.dimitrov\AppData\Roaming\Python\Python313\site-packages\joblib\__init__.py
numpy: 2.3.5 C:\Users\kamen.dimitrov\AppData\Local\miniconda3\Lib\site-packages\numpy\__init__.py
scipy: 1.17.0 C:\Users\kamen.dimitrov\AppData\Local\miniconda3\Lib\site-packages\scipy\__init__.py


In [1]:
#Import your libraries here

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load saved fitted model artifact

The training notebook should have saved this file after fitting the KMeans model.

In [2]:
# Import project modules and load trained artifact

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import (
    build_credit_report_tables,
    save_credit_report_outputs,
    save_credit_pdf_report,
)

from src.credit_clustering import (
    # Artifact utilities
    load_artifact,
    get_segment_artifact,
    infer_near_default_cluster_from_artifact,

    # Scoring and diagnostics
    score_companies,
    add_adjacent_bucket_distances_and_outlook,
    compare_to_cluster_profile,
    make_scenarios,

    #Analyst guardrails
    apply_credit_guardrails,

    # Config
    DEFAULT_PRIMARY_SEGMENT,
    DEFAULT_SCORING_TEMPERATURE,
    DEFAULT_FX_TO_MODEL_CURRENCY,
    DEFAULT_SCORING_MIN_DENOMINATOR,
    RATIO_COLS,
    SUMMARY_COLS_WITH_OUTLOOK,
    SCENARIO_SUMMARY_COLS,
)

MODEL_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "saved_models"
    / "nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"
)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model artifact not found at {MODEL_PATH}")

INPUT_PATH = PROJECT_ROOT / "inputs"
INPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "company_scores"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

artifact = load_artifact(MODEL_PATH)

SCORING_SEGMENT = artifact.get("primary_segment", DEFAULT_PRIMARY_SEGMENT)
SCORING_TEMPERATURE = DEFAULT_SCORING_TEMPERATURE
FX_TO_MODEL_CURRENCY = DEFAULT_FX_TO_MODEL_CURRENCY
SCORING_MIN_DENOMINATOR = DEFAULT_SCORING_MIN_DENOMINATOR

NEAR_DEFAULT_CLUSTER = infer_near_default_cluster_from_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

segment_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("Scoring segment:", SCORING_SEGMENT)
print("Artifact version:", artifact.get("artifact_version"))
print("Features used by model:", segment_artifact.get("feature_cols"))
print("Cluster labels:", segment_artifact.get("cluster_labels"))
print("Risk rank:", segment_artifact.get("risk_rank"))
print("Inferred near-default cluster:", NEAR_DEFAULT_CLUSTER)

PROJECT_ROOT: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\Machine_learning\08_final_project_3
MODEL_PATH: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\Machine_learning\08_final_project_3\outputs\saved_models\nonfinancial_credit_scorecard_kmeans_k5_v3.joblib
Scoring segment: Non-financial
Artifact version: v3_scorecard
Features used by model: ['earnings_risk', 'structural_distress_risk', 'operating_cashflow_risk', 'liquidity_risk', 'debt_service_risk', 'leverage_risk']
Cluster labels: {1: '1 - Strong relative credit profile', 3: '2 - Good credit profile', 0: '3 - Leveraged / elevated risk profile', 4: '4 - Weak credit profile', 2: '5 - Distressed / near-default proxy'}
Risk rank: {1: 1, 3: 2, 0: 3, 4: 4, 2: 5}
Inferred near-default cluster: 2


C:\Users\kamen.dimitrov\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\kamen.dimitrov\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator KMeans from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\kamen.dimitrov\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline fro

## 2. Manual input example

Edit the numbers below to score a private company, competitor, or simulated case.

Currency note: if your model was trained on USD and the input is in EUR, set `FX_TO_MODEL_CURRENCY` to the EUR/USD conversion rate. Ratios are mostly unaffected; `log_assets` is affected.

In [3]:
manual_input_2025 = pd.DataFrame([{
    "company_name": "Manual 2025 Company",
    "fiscal_year": 2025,
    "currency": "USD",
    "major_sector": "Manufacturing / Industrials",
    "financial_flag": "Non-financial",

    # Values converted from thousand BGN to full USD units at 1.7 BGN/USD
    "revenue": 48_585_294.12,
    "assets": 29_721_275.23,
    "current_assets": 10_037_745.82,
    "cash": 34_804.65,
    "receivables": 2_208_235.29,
    "inventory": 7_794_705.88,
    "equity": 9_082_352.94,
    "current_liabilities": 12_722_352.94,
    "liabilities": 20_638_823.53,
    "long_term_debt": 7_910_588.24,
    "short_term_debt": 1_478_235.29,
    "net_income": 949_411.76,
    "cfo": 3_558_235.29,
    "capex": 588_235.29,
    "operating_income": 1_664_705.88,
    "interest_expense": 597_647.06,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 2_713_000,
    "ebitda": np.nan,
}])

template_path = INPUT_PATH / "model_2025_company.csv"

manual_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)


Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\Machine_learning\08_final_project_3\inputs\model_2025_company.csv


In [4]:
eeec_input_2025 = pd.DataFrame([{
    "company_name": "Eastern European Electric Company B.V.",
    "fiscal_year": 2025,
    "currency": "BGN",
    "major_sector": "Transportation / Utilities",
    "financial_flag": "Non-financial",
    "revenue": 2_572_207_000,
    "assets": 2_127_723_000,
    "current_assets": 1_024_267_000,
    "cash": 224_558_000,
    "receivables": 465_950_000,
    "inventory": 42_958_000,
    "equity": 457_686_000,
    "current_liabilities": 541_002_000,
    "liabilities": 1_670_037_000,
    "long_term_debt": 988_887_000,
    "short_term_debt": 14_753_000,
    "net_income": 36_799_000,
    "cfo": 265_267_000,
    "capex": 212_690_000,
    "operating_income": 54_961_000,
    "interest_expense": 102_302_000,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 119_519_000,
    "ebitda": 259_574_000,
}])

template_path = INPUT_PATH / "EEEC_2025.csv"

eeec_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)

Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\Machine_learning\08_final_project_3\inputs\EEEC_2025.csv


In [5]:
#Configure input data model
template_path = INPUT_PATH / "EEEC_2025.csv"
current_input = pd.read_csv(template_path)

#Configure output files common name
current_file_name = "EEEC_2025"

display(current_input)

,company_name,fiscal_year,currency,major_sector,financial_flag,revenue,assets,current_assets,cash,receivables,inventory,equity,current_liabilities,liabilities,long_term_debt,short_term_debt,net_income,cfo,capex,operating_income,interest_expense,depreciation_amortization,ebitda
0,Eastern European Electric Company B.V.,2025,BGN,Transportation / Utilities,Non-financial,2572207000,2127723000,1024267000,224558000,465950000,42958000,457686000,541002000,1670037000,988887000,14753000,36799000,265267000,212690000,54961000,102302000,119519000,259574000


In [6]:
scored_manual = score_companies(
    input_df=current_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=DEFAULT_SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_manual_with_outlook = add_adjacent_bucket_distances_and_outlook(
    scored_manual,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

current_external_rating = None # optional, none is not existant
scored_manual_with_outlook = apply_credit_guardrails(
    scored_manual_with_outlook,
    external_rating=current_external_rating
)

existing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col in scored_manual_with_outlook.columns
]

missing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col not in scored_manual_with_outlook.columns
]

if missing_summary_cols_with_outlook:
    print("Missing summary columns:", missing_summary_cols_with_outlook)

display(scored_manual_with_outlook[existing_summary_cols_with_outlook])

AttributeError: 'SimpleImputer' object has no attribute '_fill_dtype'

In [ ]:
# ---------------------------------------------------------------------
# 8. Company ratio and diagnostic snapshot
# ---------------------------------------------------------------------

ratio_cols = artifact.get("ratio_cols", RATIO_COLS)

existing_ratio_cols = [
    col for col in ratio_cols
    if col in scored_manual.columns
]

missing_ratio_cols = [
    col for col in ratio_cols
    if col not in scored_manual.columns
]

if missing_ratio_cols:
    print("Missing ratio/diagnostic columns:", missing_ratio_cols)

company_ratio_snapshot = scored_manual[
    ["company_name"] + existing_ratio_cols
].copy()

display(company_ratio_snapshot.T.round(4))

## 3. Compare the company to cluster medians

This helps explain why the company was assigned to a specific profile.

In [ ]:
comparison_to_cluster = compare_to_cluster_profile(
    scored_manual.iloc[0],
    artifact,
)

comparison_to_cluster

## 4. Scenario analysis

This section shows how a company migrates across clusters under stress cases

In [ ]:
# ---------------------------------------------------------------------
# 10. Build scenario inputs
# ---------------------------------------------------------------------

scenario_input = make_scenarios(current_input.iloc[0])

scenario_input_cols = [
    "scenario",
    "assets",
    "liabilities",
    "equity",
    "cash",
    "revenue",
    "net_income",
    "cfo",
    "long_term_debt",
    "short_term_debt",
    "operating_income",
    "interest_expense",
    "depreciation_amortization",
    "ebitda",
]

existing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col in scenario_input.columns
]

missing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col not in scenario_input.columns
]

if missing_scenario_input_cols:
    print("Missing scenario input columns:", missing_scenario_input_cols)

display(scenario_input[existing_scenario_input_cols])

In [ ]:
# ---------------------------------------------------------------------
# 11. Score scenarios
# ---------------------------------------------------------------------

scored_scenarios = score_companies(
    input_df=scenario_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_scenarios = add_adjacent_bucket_distances_and_outlook(
    scored_scenarios,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

scored_scenarios = apply_credit_guardrails(
    scored_scenarios,
    external_rating=current_external_rating,
)

scenario_summary_cols = artifact.get("scenario_summary_cols", SCENARIO_SUMMARY_COLS)

# Add selected diagnostic ratios for direct scenario comparison.
scenario_diagnostic_cols = [
    "liabilities_to_assets",
    "debt_to_assets",
    "debt_to_equity",
    "equity_to_assets",
    "cfo_to_assets",
    "current_ratio",
    "quick_ratio",
    "interest_coverage",
    "fcf_to_debt",
    "ebitda",
    "ebitda_margin",
    "debt_to_ebitda",
    "net_debt_to_ebitda",
    "ebitda_interest_coverage",
    "debt_service_risk",
]

scenario_display_cols = []

for col in list(scenario_summary_cols) + scenario_diagnostic_cols:
    if col in scored_scenarios.columns and col not in scenario_display_cols:
        scenario_display_cols.append(col)

missing_scenario_cols = [
    col for col in list(scenario_summary_cols) + scenario_diagnostic_cols
    if col not in scored_scenarios.columns
]

if missing_scenario_cols:
    print("Missing scenario columns:", missing_scenario_cols)

display(scored_scenarios[scenario_display_cols])

## 5. Save report outputs to files


In [ ]:
# ---------------------------------------------------------------------
# 12. Build and save credit report outputs
# ---------------------------------------------------------------------

report_tables = build_credit_report_tables(
    scored_manual_with_outlook=scored_manual_with_outlook,
    scored_manual=scored_manual,
    comparison_to_cluster=comparison_to_cluster,
    scenario_input=scenario_input,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
)

saved_paths = save_credit_report_outputs(
    tables=report_tables,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("Scores saved to:", saved_paths["score_csv"])
print("Scenario CSV saved to:", saved_paths["scenario_csv"])
print("Full Excel score report saved to:", saved_paths["report_xlsx"])

display(report_tables["scored_file"])
display(report_tables["scenario_file"])

In [ ]:
guardrail_cols = [
    "guardrail_level",
    "guardrail_flags",
    "guardrail_summary",
    "analyst_interpretation",
    "commercial_conclusion",
]

display(scored_manual_with_outlook[guardrail_cols])

In [ ]:
pdf_path = save_credit_pdf_report(
    report_tables=report_tables,
    scored_manual_with_outlook=scored_manual_with_outlook,
    comparison_to_cluster=comparison_to_cluster,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("PDF credit report saved to:", pdf_path)

## 6. Optional: inspect saved artifact internals

In [ ]:
# ---------------------------------------------------------------------
# Optional artifact inspection
# ---------------------------------------------------------------------

nonfin_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("Artifact version:")
print(artifact.get("artifact_version"))

print("\nPrimary segment:")
print(artifact.get("primary_segment"))

print("\nFeatures used by model:")
print(nonfin_artifact.get("feature_cols"))

print("\nCluster labels:")
print(nonfin_artifact.get("cluster_labels"))

print("\nRisk rank:")
print(nonfin_artifact.get("risk_rank"))

print("\nCluster sizes:")
display(nonfin_artifact.get("cluster_sizes"))

print("\nCluster profile:")
display(nonfin_artifact.get("cluster_profile"))

print("\nCluster profile ranked:")
display(nonfin_artifact.get("cluster_profile_ranked"))